In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

In [7]:
file_name = '../dataset/kidney.csv'
df = pd.read_csv(file_name)

print("--- Data Head ---")
print(df.head())


--- Data Head ---
   age  blood_pressure  specific_gravity  albumin  sugar red_blood_cells  \
0   55             124             1.009        0      1        abnormal   
1   37             103             1.016        3      4          normal   
2   42              87             1.009        0      4          normal   
3   79              90             1.018        2      0          normal   
4   37             150             1.008        5      3        abnormal   

   pus_cell pus_cell_clumps    bacteria  blood_glucose_random  ...  \
0    normal      notpresent     present                   323  ...   
1  abnormal         present     present                    70  ...   
2    normal      notpresent  notpresent                   441  ...   
3  abnormal         present  notpresent                   449  ...   
4  abnormal      notpresent     present                   180  ...   

   packed_cell_volume  white_blood_cell_count  red_blood_cell_count  \
0                  27            

In [8]:
print("\n--- Data Info ---")
df.info()


--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      500 non-null    int64  
 1   blood_pressure           500 non-null    int64  
 2   specific_gravity         500 non-null    float64
 3   albumin                  500 non-null    int64  
 4   sugar                    500 non-null    int64  
 5   red_blood_cells          500 non-null    object 
 6   pus_cell                 500 non-null    object 
 7   pus_cell_clumps          500 non-null    object 
 8   bacteria                 500 non-null    object 
 9   blood_glucose_random     500 non-null    int64  
 10  blood_urea               500 non-null    float64
 11  serum_creatinine         500 non-null    float64
 12  sodium                   500 non-null    float64
 13  potassium                500 non-null    float64
 14  haemogl

In [9]:

mapping_dict = {
    'normal': 0, 'abnormal': 1,
    'notpresent': 0, 'present': 1,
    'no': 0, 'yes': 1,
    'poor': 0, 'good': 1
}

object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].map(mapping_dict)

print("\n--- Data Info After Preprocessing ---")
df.info()


--- Data Info After Preprocessing ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      500 non-null    int64  
 1   blood_pressure           500 non-null    int64  
 2   specific_gravity         500 non-null    float64
 3   albumin                  500 non-null    int64  
 4   sugar                    500 non-null    int64  
 5   red_blood_cells          500 non-null    int64  
 6   pus_cell                 500 non-null    int64  
 7   pus_cell_clumps          500 non-null    int64  
 8   bacteria                 500 non-null    int64  
 9   blood_glucose_random     500 non-null    int64  
 10  blood_urea               500 non-null    float64
 11  serum_creatinine         500 non-null    float64
 12  sodium                   500 non-null    float64
 13  potassium                500 non-null    

In [10]:

X = df.drop('target', axis=1)
y = df['target']


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

Training set shape: (400, 24)
Test set shape: (100, 24)


In [12]:

binary_cols = list(object_cols) 
numerical_cols = [col for col in X_train.columns if X_train[col].nunique() > 2]

print(f"\nColumns to be scaled: {numerical_cols}")

scaler = StandardScaler()

scaler.fit(X_train[numerical_cols])

X_train[numerical_cols] = scaler.transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("\n--- Training Data Head After Scaling ---")
print(X_train.head())


Columns to be scaled: ['age', 'blood_pressure', 'specific_gravity', 'albumin', 'sugar', 'blood_glucose_random', 'blood_urea', 'serum_creatinine', 'sodium', 'potassium', 'haemoglobin', 'packed_cell_volume', 'white_blood_cell_count', 'red_blood_cell_count']

--- Training Data Head After Scaling ---
          age  blood_pressure  specific_gravity   albumin     sugar  \
19   1.567973       -1.690459          1.309639  1.428759  0.241599   
182 -1.256326        0.529656         -1.407454  0.231893 -0.358647   
284  1.178415        1.423468         -0.320617 -0.964973 -0.958894   
447 -0.525904       -0.767814          1.490778 -1.563406 -1.559141   
297 -0.038956        0.558489          1.128499 -1.563406 -0.358647   

     red_blood_cells  pus_cell  pus_cell_clumps  bacteria  \
19                 1         1                1         0   
182                0         0                0         0   
284                0         1                1         1   
447                0         0

In [13]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

print("KNN model trained successfully.")

KNN model trained successfully.


In [14]:
y_pred = knn.predict(X_test)

In [15]:
accuracy = accuracy_score(y_test, y_pred)
print(f"--- Model Evaluation (n_neighbors=5) ---")
print(f"\nAccuracy: {accuracy:.4f}")

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

--- Model Evaluation (n_neighbors=5) ---

Accuracy: 0.5300

--- Confusion Matrix ---
[[38 24]
 [23 15]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.62      0.61      0.62        62
           1       0.38      0.39      0.39        38

    accuracy                           0.53       100
   macro avg       0.50      0.50      0.50       100
weighted avg       0.53      0.53      0.53       100



In [16]:
import pickle


In [17]:
filename = 'kidney.sav'
pickle.dump(knn, open(filename, 'wb'))